# Use the Schema Registry

This guide shows how to register configuration schemas and use the `SchemaRegistry` to validate configuration and generate JSON Schemas.

## Create a Registry

When creating a new schema registry it will be empty. Schemas can be registered manually or automatically by using the discovery functionality which will be explained later in this guide.

In [1]:
from pyaml.validation import SchemaRegistry

registry = SchemaRegistry()
print(registry)

SchemaRegistry({})


The registry is implemented as a singleton which means the same object will be returned every time you create a new instance of it during the same session. Schemas therefore only have to be registered once per session.

## Register a Schema

There are two ways to register a schema:

1. Use the `register` method
2. Use the `register_schema` decorator to automatically register schemas when a module is loaded

    The decorator can be used in two ways: dynamically generating the schema from the class constructor or by explicitly declaring a schema too use for the class.

    Note: a decorator only runs when a module is imported. If you use the `register_schema` decorator in a module that is not imported the schema will not be registered.

The different ways to use the decorator is explained below.

### Register a Dynamic Schema

The decorator generates a `ConfigurationSchema` and registers it in the schema registry using the class's fully qualified path.

In [2]:
from pyaml.validation import register_schema

@register_schema
class Magnet:
    def __init__(self, length: float):
        self.length = length

The schema is now visible in the registry. In this case the schema will get the path `__main__.Magnet` since the class was declared directly in the script. For other classes it will be of the form `package.module.Class`.

You can use `describe` to get pretty output of the fields in the schema. `ConfigurationSchema` inherits from Pydantic `BaseModel` so the functionality of that class is also available. See Pydantic's documentation for details.

In [3]:
# See the content of the registry
print(registry)

# See the fields in the model
print(registry["__main__.Magnet"].describe())

SchemaRegistry(
    '__main__.Magnet': __main__.MagnetConfigurationSchema,
)
MagnetConfigurationSchema(
    class_path: str — Fully qualified class path.
    length: float
)


### Register an Explicit Schema

Define a `ConfigurationSchema` explicitly when you need more control over the configuration fields or validation rules. The functionality of Pydantic is available when defining the schema.

The explicit schema must inherit from `ConfigurationSchema` since that defined the minimum required fields for all items in the registry.

In [4]:
from pyaml.validation import ConfigurationSchema, register_schema

class DifferentMagnetSchema(ConfigurationSchema):
    length: float

@register_schema(DifferentMagnetSchema)
class DifferentMagnet:
    def __init__(self, length: float):
        self.length = length

Bot schemas are now available in the registry.

In [5]:
print(registry)

SchemaRegistry(
    '__main__.DifferentMagnet': __main__.DifferentMagnetSchema,
    '__main__.Magnet': __main__.MagnetConfigurationSchema,
)


## Automatically Discover Schemas

Since registration only happens when the modules containing decorated classes are imported, a `discover` method is available to automatically scan packages and register the schemas in the package.

In [6]:
from pyaml.validation import SchemaRegistry

registry = SchemaRegistry()

# Clear the registry to remove the schemas that were manually registered
registry.clear()

# Automatically discover and register all schemas
registry.discover()

# Print the content of the registry
print(registry)

SchemaRegistry(
    'abc.ABC': abc.ABCConfigurationSchema,
    'pyaml.arrays.array.ArrayConfig': pyaml.arrays.array.ArrayConfigConfigurationSchema,
    'pyaml.arrays.bpm.BPM': pyaml.arrays.bpm.BPMConfigurationSchema,
    'pyaml.arrays.cfm_magnet.CombinedFunctionMagnet': pyaml.arrays.cfm_magnet.CombinedFunctionMagnetConfigurationSchema,
    'pyaml.arrays.element.Element': pyaml.arrays.element.ElementConfigurationSchema,
    'pyaml.arrays.magnet.Magnet': pyaml.arrays.magnet.MagnetConfigurationSchema,
    'pyaml.arrays.serialized_magnet.SerializedMagnets': pyaml.arrays.serialized_magnet.SerializedMagnetsConfigurationSchema,
    'pyaml.bpm.bpm.BPM': pyaml.bpm.bpm.BPMConfigurationSchema,
    'pyaml.common.element.Element': pyaml.common.element.ElementConfigurationSchema,
    'pyaml.common.holders.element_holder.ElementHolder': pyaml.common.holders.element_holder.ElementHolderConfigurationSchema,
    'pyaml.diagnostics.atune_monitor.ABetatronTuneMonitor': pyaml.diagnostics.atune_monitor.ABet

## Browse the Registry

The registry also supports common mapping operations. Class paths are the keys and registered schema classes are the values. See the API documentation for all available methods.

Use `get()` to retrieve the schema for a class path. It returns `None` when the path is not registered.

In [7]:
schema = registry.get("pyaml.magnet.quadrupole.Quadrupole")
print(schema.describe())

QuadrupoleConfigurationSchema(
    class_path: str — Fully qualified class path.
    name: str
    model: pyaml.magnet.model.MagnetModelConfigurationSchema | None
    lattice_names: str | None
    description: str | None
)


In [8]:
class_path = "pyaml.magnet.quadrupole.Quadrupole"

# Check if the class is in the registry
print(class_path in registry)

True


In [9]:
# Iterate through the registered schemas

print("\nRegistered class paths:")
for path in registry:
    print(f"- {path}")

print("\n Registered schema classes:")
for schema in registry.values():
    print(f"- {schema.__name__}")    


Registered class paths:
- pyaml.bpm.bpm.BPM
- pyaml.common.element.Element
- pyaml.validation.validation_models.DynamicValidation
- pyaml.magnet.model.MagnetModel
- pyaml.magnet.hcorrector.HCorrector
- pyaml.magnet.magnet.Magnet
- pyaml.magnet.octupole.Octupole
- pyaml.magnet.quadrupole.Quadrupole
- pyaml.magnet.sextupole.Sextupole
- pyaml.magnet.skewoctu.SkewOctu
- pyaml.magnet.skewquad.SkewQuad
- pyaml.magnet.skewsext.SkewSext
- pyaml.magnet.vcorrector.VCorrector
- typing.Any
- pyaml.magnet.cfm_magnet.CombinedFunctionMagnet
- pyaml.magnet.serialized_magnet.SerializedMagnets
- pyaml.diagnostics.tune_monitor.BetatronTuneMonitor
- pyaml.diagnostics.atune_monitor.ABetatronTuneMonitor
- pyaml.rf.rf_transmitter.RFTransmitter
- pyaml.rf.rf_plant.RFPlant
- pyaml.tuning_tools.chromaticity_monitor.ChromaticityMonitor
- pyaml.tuning_tools.measurement_tool.MeasurementTool
- pyaml.arrays.array.ArrayConfig
- pyaml.lattice.lattice_elements_linker.LinkerConfigModel
- abc.ABC
- pyaml.lattice.lattice

In [10]:
# Print the number of schemas in the registry
print(f"\nNumber of registered schemas: {len(registry)}")


Number of registered schemas: 58


## Validate Configuration

Configuration data can be validated using the `SchemaValidator`. It makes use of the schema registry to extract which schema to validate against for a specific class.

For validation to be possible the class must be registered in the schema registry. If the class is not registered, validation will be skipped, a warning given and the data kept unchanged. Beware that this can lead to unexpected errors.

In [11]:
model_path = "pyaml.magnet.identity_model.IdentityMagnetModel"
print(registry.get(model_path))

<class 'pyaml.magnet.identity_model.IdentityMagnetModelConfigurationSchema'>


In [12]:
from pyaml.validation import SchemaRegistry, SchemaValidator

registry = SchemaRegistry()
registry.discover()

configuration = {
    "class_path": "pyaml.magnet.quadrupole.Quadrupole",
    "name": "QF1",
    "description": "This is the QF1 quadrupole magnet."
}
validated = SchemaValidator.validate(configuration)

Validation also handles nested configuration data.

In [13]:
configuration = {
    "class_path": "pyaml.magnet.quadrupole.Quadrupole",
    "name": "QF1",
    "model": {
        "class_path": "pyaml.magnet.identity_model.IdentityMagnetModel",
        "unit": "1/m",
        "physics": ""
    },
    "description": "This is the QF1 quadrupole magnet."
}

validated = SchemaValidator.validate(configuration)
print(validated)

class_path='pyaml.magnet.quadrupole.Quadrupole' name='QF1' model=IdentityMagnetModelConfigurationSchema(class_path='pyaml.magnet.identity_model.IdentityMagnetModel', powerconverter=None, physics='', unit='1/m') lattice_names=None description='This is the QF1 quadrupole magnet.'


The validated result can also be returned as a dictionary.

In [14]:
from pprint import pprint

validated_dict = SchemaValidator.validate_to_dict(configuration)
pprint(validated_dict )

{'class_path': 'pyaml.magnet.quadrupole.Quadrupole',
 'description': 'This is the QF1 quadrupole magnet.',
 'lattice_names': None,
 'model': {'class_path': 'pyaml.magnet.identity_model.IdentityMagnetModel',
           'physics': '',
           'powerconverter': None,
           'unit': '1/m'},
 'name': 'QF1'}
